# SESIÓN 3: CREACIÓN Y MANIPULACIÓN DE TABLAS (PARTE III)
## Fundamentos de Programación Python para el Análisis de Datos

Actualización de registros usando UPDATE

In [ ]:
import psycopg2

# Conexión a base de datos
conn = psycopg2.connect(
    host='localhost',
    database='capacitaciones',
    user='postgres',
    password='password'
)
cur = conn.cursor()

## SLIDE 5: Sintaxis básica de UPDATE

In [ ]:
# Sintaxis básica de UPDATE
sql_sintaxis = """
UPDATE nombre_tabla
SET columna1 = valor1 [, columna2 = valor2, ...]
WHERE condición;
"""
print(sql_sintaxis)

## SLIDE 6: Ejemplos básicos de UPDATE

In [ ]:
# Crear tabla de cursos para ejemplos
sql_create = """
DROP TABLE IF EXISTS cursos CASCADE;
CREATE TABLE cursos (
    id_curso SERIAL PRIMARY KEY,
    nombre_curso VARCHAR(100) NOT NULL,
    duracion INT DEFAULT 20 CHECK (duracion > 0),
    modalidad VARCHAR(20) DEFAULT 'Online' CHECK (modalidad IN ('Online', 'Presencial'))
)
"""

cur.execute(sql_create)
conn.commit()

# Insertar datos iniciales
sql_insert = """
INSERT INTO cursos (nombre_curso, duracion, modalidad)
VALUES 
    ('SQL aplicado a negocios', 25, 'Presencial'),
    ('Python para principiantes', 30, 'Online'),
    ('Power BI básico', 20, 'Presencial')
"""

cur.execute(sql_insert)
conn.commit()
print("Tabla de cursos creada con datos iniciales")

In [ ]:
# Ejemplo 1: Actualizar un campo específico
sql_update_1 = """
UPDATE cursos
SET duracion = 30
WHERE nombre_curso = 'SQL aplicado a negocios';
"""

cur.execute(sql_update_1)
conn.commit()
print("✓ Actualización 1: Duración de SQL aumentada a 30")

In [ ]:
# Ejemplo 2: Actualizar con operación aritmética
sql_update_2 = """
UPDATE cursos
SET duracion = duracion + 10
WHERE modalidad = 'Presencial';
"""

cur.execute(sql_update_2)
conn.commit()
print("✓ Actualización 2: Duración de cursos presenciales aumentada en 10 horas")

## SLIDE 8: Importancia de la cláusula WHERE

In [ ]:
# ADVERTENCIA: Ejemplo de lo que NO debes hacer
# Si ejecutas esto SIN WHERE, actualizarás TODOS los cursos

sql_peligroso = """
-- UPDATE cursos
-- SET modalidad = 'Presencial';
-- ¡Esto actualizaría TODOS los cursos!
"""

print("ADVERTENCIA: Nunca uses UPDATE sin WHERE (a menos que sea intencional)")
print("Siempre verifica qué registros serán afectados con SELECT primero")

In [ ]:
# BUENA PRÁCTICA: Verificar primero con SELECT
sql_verificar = """
SELECT * FROM cursos
WHERE modalidad = 'Online';
"""

cur.execute(sql_verificar)
registros = cur.fetchall()

print(f"Registros a ser modificados: {len(registros)}")
for reg in registros:
    print(f"  ID: {reg[0]}, Nombre: {reg[1]}, Duración: {reg[2]}, Modalidad: {reg[3]}")

## SLIDE 9: Condiciones complejas con AND/OR

In [ ]:
# Ejemplo: Condición compuesta con AND
sql_update_compleja = """
UPDATE cursos
SET modalidad = 'Presencial'
WHERE duracion > 25 
  AND modalidad = 'Online';
"""

# Primero verificamos qué se va a actualizar
sql_check = """
SELECT * FROM cursos
WHERE duracion > 25 AND modalidad = 'Online';
"""

cur.execute(sql_check)
result = cur.fetchall()

print(f"Registros que cumplen la condición: {len(result)}")
for reg in result:
    print(f"  {reg[1]} - Duración: {reg[2]}, Modalidad: {reg[3]}")

## SLIDE 10: Actualización de múltiples columnas

In [ ]:
# Actualizar múltiples campos en una sola instrucción
sql_update_multiple = """
UPDATE cursos
SET duracion = 40, 
    modalidad = 'Presencial'
WHERE nombre_curso = 'Power BI básico';
"""

cur.execute(sql_update_multiple)
conn.commit()
print("✓ Actualización múltiple: Power BI ahora tiene duración 40 y modalidad Presencial")

## SLIDE 13: Validación de restricciones

In [ ]:
# Ejemplo: Intentar violar una restricción CHECK
sql_error_check = """
UPDATE cursos
SET modalidad = 'Virtual'
WHERE id_curso = 1;
"""

try:
    cur.execute(sql_error_check)
    conn.commit()
    print("✓ Actualización realizada")
except Exception as e:
    conn.rollback()
    print(f"✗ Error CHECK: La modalidad 'Virtual' no es permitida")
    print(f"  Valores permitidos: 'Online' o 'Presencial'")

In [ ]:
# Ejemplo: Intentar violar una restricción CHECK (duración negativa)
sql_error_duracion = """
UPDATE cursos
SET duracion = -5
WHERE id_curso = 2;
"""

try:
    cur.execute(sql_error_duracion)
    conn.commit()
    print("✓ Actualización realizada")
except Exception as e:
    conn.rollback()
    print(f"✗ Error CHECK: La duración debe ser positiva")
    print(f"  No se permite valores negativos o cero")

## SLIDE 19: Actividad guiada - Corrección de registros

In [ ]:
# Crear tabla para actividad guiada
sql_actividad = """
DROP TABLE IF EXISTS cursos_actividad CASCADE;
CREATE TABLE cursos_actividad (
    id_curso INT PRIMARY KEY,
    nombre_curso VARCHAR(50) NOT NULL,
    duracion INT CHECK (duracion > 0),
    modalidad VARCHAR(20) CHECK (modalidad IN ('Online', 'Presencial'))
);

INSERT INTO cursos_actividad (id_curso, nombre_curso, duracion, modalidad) 
VALUES
    (1, 'Excel básico', 20, 'Presencial'),
    (2, 'Análisis de datos', 25, 'Presencial'),
    (3, 'SQL aplicado a negocios', 10, 'Presencial'),
    (4, 'Python para principiantes', 30, 'Onlline');
"""

cur.execute(sql_actividad)
conn.commit()
print("Tabla de actividad creada con datos iniciales")

In [ ]:
# Ver datos iniciales
sql_ver = """
SELECT * FROM cursos_actividad;
"""

cur.execute(sql_ver)
registros = cur.fetchall()

print("Datos iniciales:")
print("-" * 70)
for reg in registros:
    print(f"ID: {reg[0]} | Nombre: {reg[1]} | Duración: {reg[2]} | Modalidad: {reg[3]}")

In [ ]:
# CORRECCIÓN 1: Corregir modalidad mal escrita (Onlline -> Online)
# Nota: Esta tabla tiene error tipográfico 'Onlline' que no cumple CHECK
# En la realidad, no se habría permitido insertar, pero lo simulamos así

sql_correccion_1 = """
UPDATE cursos_actividad
SET modalidad = 'Online'
WHERE id_curso = 4;
"""

try:
    cur.execute(sql_correccion_1)
    conn.commit()
    print("✓ Corrección 1: Modalidad del curso 4 actualizada a 'Online'")
except Exception as e:
    conn.rollback()
    print(f"Error: {e}")

In [ ]:
# CORRECCIÓN 2: Ajustar duración incorrecta
sql_correccion_2 = """
UPDATE cursos_actividad
SET duracion = 30
WHERE nombre_curso = 'SQL aplicado a negocios';
"""

try:
    cur.execute(sql_correccion_2)
    conn.commit()
    print("✓ Corrección 2: Duración de 'SQL aplicado a negocios' actualizada a 30")
except Exception as e:
    conn.rollback()
    print(f"Error: {e}")

In [ ]:
# Ver datos después de correcciones
cur.execute(sql_ver)
registros = cur.fetchall()

print("Datos después de correcciones:")
print("-" * 70)
for reg in registros:
    print(f"ID: {reg[0]} | Nombre: {reg[1]} | Duración: {reg[2]} | Modalidad: {reg[3]}")

## SLIDE 23: Actividad autónoma - Corrección de datos erróneos

In [ ]:
# Crear tabla para actividad autónoma
sql_autonoma = """
DROP TABLE IF EXISTS programas CASCADE;
CREATE TABLE programas (
    id_programa INT PRIMARY KEY,
    nombre_programa VARCHAR(50) NOT NULL,
    duracion INT CHECK (duracion > 0),
    modalidad VARCHAR(20) CHECK (modalidad IN ('Online', 'Presencial'))
);

INSERT INTO programas (id_programa, nombre_programa, duracion, modalidad) 
VALUES
    (1, 'Data Science Inicial', 60, 'Presencial'),
    (2, 'Inteligencia Artificial', 40, 'Presencial'),
    (3, 'Big Data', 15, 'Presencial'),
    (4, 'Análisis de Datos con Python', 45, 'Onlne');
"""

cur.execute(sql_autonoma)
conn.commit()
print("Tabla de programas creada")

In [ ]:
# Ver datos iniciales
sql_programas = """
SELECT * FROM programas;
"""

cur.execute(sql_programas)
registros = cur.fetchall()

print("Datos iniciales de programas:")
print("-" * 70)
for reg in registros:
    print(f"ID: {reg[0]} | Programa: {reg[1]} | Duración: {reg[2]} | Modalidad: {reg[3]}")

In [ ]:
# TAREA 1: Corregir modalidad mal escrita (Onlne -> Online)
sql_corr_autonoma_1 = """
UPDATE programas
SET modalidad = 'Online'
WHERE id_programa = 4;
"""

try:
    cur.execute(sql_corr_autonoma_1)
    conn.commit()
    print("✓ Tarea 1 completada: Modalidad corregida")
except Exception as e:
    conn.rollback()
    print(f"Error: {e}")

In [ ]:
# TAREA 2: Ajustar duración de 'Big Data' a 30 horas
sql_corr_autonoma_2 = """
UPDATE programas
SET duracion = 30
WHERE nombre_programa = 'Big Data';
"""

try:
    cur.execute(sql_corr_autonoma_2)
    conn.commit()
    print("✓ Tarea 2 completada: Duración de Big Data actualizada")
except Exception as e:
    conn.rollback()
    print(f"Error: {e}")

In [ ]:
# Ver datos después de correcciones
cur.execute(sql_programas)
registros = cur.fetchall()

print("Datos después de correcciones:")
print("-" * 70)
for reg in registros:
    print(f"ID: {reg[0]} | Programa: {reg[1]} | Duración: {reg[2]} | Modalidad: {reg[3]}")

## Cerrar conexión

In [ ]:
cur.close()
conn.close()
print("Conexión cerrada")